# 01. Lists

**Tier:** Core. **Why you'd use this:** A packing list grows as you append items. Two names can point at the same list, so an edit through one name shows up through the other. You also need to know why adding at the end is cheap and inserting at the front is not.

**After this lesson you can:** tell a change to the same list from building a new list; explain why `append` is cheap and `insert(0)` is not; read an `IndexError`.
About 25 minutes. **Uses:** [Loops](../03-control-flow/02-loops-lesson.ipynb)


## Quick review

1. What does `for n in range(3):` bind `n` to on each pass?
2. How do you stop a loop early when you already have the answer?

<details><summary>Answers</summary>

`0`, then `1`, then `2`. Use `break`.
</details>


## The idea

A list holds values in order, and you can change that same list. `append` adds at the end. `sorted` returns a new list and leaves the original alone.

<details><summary>Explain more</summary>

`nums[0]` is the first item. `pop()` removes and returns the last. `append` returns `None` because it changes the list in place.
</details>


## See it work

Keep only the positive numbers.


In [1]:
def positives(nums):
    # 1. Start with an empty result. This is a different list.
    kept = []
    # 2. Keep each value that is greater than zero.
    for value in nums:
        if value > 0:
            kept.append(value)
    return kept


nums = [3, -1, 2]
print(positives(nums), nums)

[3, 2] [3, -1, 2]


## Check yourself

Answer from the example before you run.


1. After `sorted(nums)`, why is `nums` still in the old order?
2. Why can `append` be cheap most of the time and suddenly copy the whole list?

<details><summary>Answers</summary>

`sorted` allocates a new list. The copy happens when spare slots run out, not on every append.
</details>


In [2]:
nums = [3, 1]
print(nums.append(2))
print(nums)
print(sorted(nums))
print(nums)

None
[3, 1, 2]
[1, 2, 3]
[3, 1, 2]


<details><summary>Check your answers</summary>

`append` returns `None` and changes `nums` to `[3, 1, 2]`. `sorted` prints `[1, 2, 3]` without writing that order back into `nums`.
</details>


## Why it works this way

A list is a dynamic array: one block of pointers plus a count in use. `append` writes into the next free slot — O(1) until the block is full, then Python allocates a larger block and copies. Over a long run the copies add up to about one copy per item ("amortized O(1)"). Spare slots are the memory cost.

`insert(0, item)` shifts every pointer — O(n) each, quadratic in a loop. `in` walks and compares, also O(n). Append stays cheap because a resize copies occasionally. Deep dive: [How list works](../../internals/03-lists/01-dynamic-array-lesson.ipynb). Indexing a slot stays O(1) because the pointers sit in one contiguous block — that is the trade for expensive inserts at the front.


In [3]:
import sys

nums = []
previous = 0
for i in range(40):
    nums.append(i)
    size = sys.getsizeof(nums)
    if size != previous:
        print(i, size)
        previous = size

0 88
4 120
8 184
16 248
24 312
32 376


In [4]:
import timeit

print(
    "append",
    timeit.timeit("nums.append(1)", setup="nums = []", number=20000),
)
print(
    "insert",
    timeit.timeit("nums.insert(0, 1)", setup="nums = []", number=20000),
)

append 0.00027249997947365046
insert 0.03688049997435883


## When to use it

Use a list when order and position matter; a dict or set later for membership. Grow with `append`; don't `insert(0)` once per item.


## Common mistake

Assigning the result of `append`. The method changes the list and returns `None`. The name now points at `None`, and the list you built is left with no name.


In [5]:
nums = [1, 2]
nums = nums.append(3)
print(nums)

None


Call `append` and keep the same name. The list object is what changed. There is no new list to store.


In [6]:
nums = [1, 2]
nums.append(3)
print(nums)

[1, 2, 3]


An index past the end is a different bug. Read the last line first: `IndexError`. The arrow marks the lookup. `len(nums)` is one past the last slot.


In [7]:
def last(nums):
    return nums[len(nums)]


print(last([1, 2]))

IndexError: list index out of range

## Try it

Change `positives` so it also returns how many values it dropped. Why can't you read that count off `nums` after the call returns?


## Recap

- A list is a block of pointers. `append` fills spare slots, then occasionally copies.
- `insert(0)` and `in` walk the block.
- `append` returns `None`. An index past the end raises `IndexError`.

Next: [practice](01-lists-practice.ipynb).
